In [ ]:
# @title **CELDA 1: GENERACIÓN DE DATOS SINTÉTICOS**

import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta
import random
from collections import defaultdict

print("🔧 CELDA 1: GENERACIÓN DE DATOS SINTÉTICOS CON MÁS BALANCES NEGATIVOS")
print("="*80)

# ===============================
# CONFIGURACIÓN
# ===============================
INPUT_DIR = "input"
OUTPUT_DIR = "output"
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

np.random.seed(42)
random.seed(42)

# ===============================
# CARGAR LISTA MAESTRA
# ===============================
lista_maestra = pd.read_csv("Lista Maestra de Residuos.csv")
residuos = lista_maestra["Residuo"].unique()
tipos_residuos = dict(zip(lista_maestra["Residuo"], lista_maestra["Tipo"]))
destinos_residuos = dict(zip(lista_maestra["Residuo"], lista_maestra["Destino"]))

print(f"📋 Lista Maestra cargada: {len(residuos)} tipos de residuos")
print(f"🔍 Distribución: {len(lista_maestra[lista_maestra['Tipo']=='Peligroso'])} peligrosos, "
      f"{len(lista_maestra[lista_maestra['Tipo']=='No Peligroso'])} no peligrosos")

# ===============================
# FUNCIONES AUXILIARES
# ===============================
def generar_fecha_aleatoria_2025():
    inicio = datetime(2025, 1, 1, 0, 0)
    fin = datetime(2025, 12, 31, 23, 59)
    delta = fin - inicio
    return inicio + timedelta(seconds=random.randint(0, int(delta.total_seconds())))

def aplicar_error_pesaje(cantidad_real, tipo_error):
    """Aplica diferentes tipos de errores de pesaje - VERSIÓN CON MÁS SUBESTIMACIÓN"""
    if tipo_error == "sobreestimacion":
        # Error: se pesa de más (5-15%)
        error = random.uniform(1.05, 1.15)
        return cantidad_real * error
    elif tipo_error == "subestimacion":
        # Error: se pesa de menos (10-30%) - AUMENTADO para más balances negativos
        error = random.uniform(0.70, 0.90)
        return cantidad_real * error
    elif tipo_error == "error_grande":
        # Error grande (30-50%) - Mayor probabilidad de subestimación
        if random.random() < 0.7:  # 70% subestimación, 30% sobreestimación
            error = random.uniform(0.5, 0.7)  # Subestimación severa
        else:
            error = random.uniform(1.3, 1.5)  # Sobreestimación
        return cantidad_real * error
    else:
        # Sin error significativo
        return cantidad_real * random.uniform(0.98, 1.02)

def simular_cambio_clasificacion(residuo_original):
    """Simula cambio de clasificación de residuos - AUMENTADA probabilidad"""
    if random.random() < 0.15:  # 15% de probabilidad (aumentado de 10%)
        cambios_posibles = {
            "Aceite Usado": ["Residuos Con Hidrocarburos", "Grasa Residual"],
            "Carton Y/O Papel": ["No Aprovechables", "Organicos Madera"],
            "Plasticos Hdpe": ["Plasticos Ldpe", "Plasticos Pet"],
            "Metalicos Cobre": ["Metalicos Bronce", "Metalicos Aluminio"],
            "Residuos De Construccion": ["Residuos Inertes", "Escombros"],
            "Lodos De Plomo": ["Residuos Con Metales Pesados", "Lodos Industriales"],
            "Residuos Con Cianuro": ["Residuos Toxicos", "Residuos Quimicos"],
            "Caucho Neumaticos": ["Residuos de Caucho", "Neumaticos Usados"]
        }
        if residuo_original in cambios_posibles:
            return random.choice(cambios_posibles[residuo_original])
    return residuo_original

# ===============================
# 1. GENERAR INVENTARIO 2024 (STOCK INICIAL CON MÁS ERRORES)
# ===============================
print("\n📦 1. Generando Inventario 2024 (con MÁS errores de subestimación)...")

inventario_2024 = []

for residuo in residuos:
    # Generar cantidad real de stock
    if tipos_residuos[residuo] == "Peligroso":
        cantidad_real = random.uniform(2000, 15000)
    else:
        cantidad_real = random.uniform(5000, 30000)

    # Aplicar error de registro inicial - MAYOR PROBABILIDAD DE SUBESTIMACIÓN
    tipo_error_inicial = random.choices(
        ["sobreestimacion", "subestimacion", "error_grande", "normal"],
        weights=[0.2, 0.5, 0.2, 0.1]  # 50% subestimación, 20% error grande
    )[0]

    cantidad_registrada = aplicar_error_pesaje(cantidad_real, tipo_error_inicial)

    inventario_2024.append({
        "ID_Evento": f"INV-2024-{residuos.tolist().index(residuo)+1:03d}",
        "Residuo": residuo,
        "Tipo": tipos_residuos[residuo],
        "Destino": destinos_residuos[residuo],
        "FechaHora": datetime(2024, 12, 31, 23, 59),
        "Cantidad_Real_kg": round(cantidad_real, 2),
        "Cantidad_Registrada_kg": round(cantidad_registrada, 2),
        "Error_Registro_kg": round(cantidad_registrada - cantidad_real, 2),
        "Tipo_Error": tipo_error_inicial
    })

df_inv_2024 = pd.DataFrame(inventario_2024)
df_inv_2024.to_csv(f"{INPUT_DIR}/Inventario_2024.csv", index=False)

# Calcular cuántos tienen subestimación significativa (>10% error)
subestimaciones = df_inv_2024[df_inv_2024["Error_Registro_kg"] < -df_inv_2024["Cantidad_Real_kg"] * 0.1]
print(f"   ✅ Generado: {len(df_inv_2024)} registros de inventario 2024")
print(f"   📉 Subestimaciones significativas: {len(subestimaciones)} registros")

# ===============================
# 2. GENERAR INGRESOS 2025 (CON MÁS ERRORES DE SUBESTIMACIÓN)
# ===============================
print("\n📥 2. Generando Ingresos 2025 con MÁS errores de subestimación...")

ingresos_2025 = []
id_ingreso = 1
errores_registrados = []

# Definir tipos de errores - MAYOR PESO A ERRORES QUE CAUSAN BALANCES NEGATIVOS
tipos_errores = {
    "error_pesaje": 0.40,           # 40% de errores de pesaje (aumentado)
    "cambio_clasificacion": 0.20,   # 20% de cambios de clasificación (aumentado)
    "registro_fecha_erronea": 0.10, # 10% de errores en fecha
    "registro_hora_erronea": 0.10,  # 10% de errores en hora
    "división_residuos": 0.10,      # 10% de divisiones de residuos
    "sin_error": 0.10               # 10% sin error (reducido)
}

for residuo in residuos:
    # Determinar cantidad total de ingresos para este residuo
    if tipos_residuos[residuo] == "Peligroso":
        cantidad_total_real = random.uniform(10000, 50000)
        num_eventos = random.randint(8, 20)
    else:
        cantidad_total_real = random.uniform(30000, 120000)
        num_eventos = random.randint(15, 40)

    # Generar eventos individuales
    proporciones = np.random.dirichlet(np.ones(num_eventos))
    cantidades_reales = proporciones * cantidad_total_real

    for i, cantidad_real in enumerate(cantidades_reales):
        # Determinar tipo de error para este evento
        tipo_error = random.choices(
            list(tipos_errores.keys()),
            weights=list(tipos_errores.values())
        )[0]

        # Aplicar error según tipo
        cantidad_registrada = cantidad_real
        error_descripcion = "Sin error"
        residuo_registrado = residuo
        fecha_hora_real = generar_fecha_aleatoria_2025()
        fecha_hora_registrada = fecha_hora_real

        if tipo_error == "error_pesaje":
            # MAYOR PROBABILIDAD DE SUBESTIMACIÓN
            subtipo = random.choices(
                ["sobreestimacion", "subestimacion", "error_grande"],
                weights=[0.2, 0.5, 0.3]  # 50% subestimación, 30% error grande
            )[0]
            cantidad_registrada = aplicar_error_pesaje(cantidad_real, subtipo)
            error_descripcion = f"Error de pesaje ({subtipo})"

        elif tipo_error == "cambio_clasificacion":
            nuevo_residuo = simular_cambio_clasificacion(residuo)
            if nuevo_residuo != residuo:
                residuo_registrado = nuevo_residuo
                error_descripcion = f"Cambio clasificación: {residuo} → {nuevo_residuo}"

        elif tipo_error == "registro_fecha_erronea":
            # Error de ±1 a ±7 días
            dias_error = random.randint(-7, 7)
            if dias_error != 0:
                fecha_hora_registrada = fecha_hora_real + timedelta(days=dias_error)
                error_descripcion = f"Error fecha: {dias_error} días"

        elif tipo_error == "registro_hora_erronea":
            # Error de ±1 a ±4 horas
            horas_error = random.randint(-4, 4)
            if horas_error != 0:
                fecha_hora_registrada = fecha_hora_real + timedelta(hours=horas_error)
                error_descripcion = f"Error hora: {horas_error} horas"

        elif tipo_error == "división_residuos":
            # Simular que un residuo se dividió en varios (esto también puede causar subestimación)
            if cantidad_real > 1000 and random.random() < 0.3:
                num_divisiones = random.randint(2, 4)
                cantidad_registrada = cantidad_real / num_divisiones
                error_descripcion = f"División en {num_divisiones} partes"

        # Registrar error si es significativo
        if tipo_error != "sin_error" and abs(cantidad_registrada - cantidad_real) > 1:
            errores_registrados.append({
                "ID_Ingreso": id_ingreso,
                "Residuo_Real": residuo,
                "Residuo_Registrado": residuo_registrado,
                "Tipo_Error": tipo_error,
                "Descripcion": error_descripcion,
                "Cantidad_Real_kg": round(cantidad_real, 2),
                "Cantidad_Registrada_kg": round(cantidad_registrada, 2),
                "Error_kg": round(cantidad_registrada - cantidad_real, 2)
            })

        ingresos_2025.append({
            "ID_Evento": f"ING-2025-{id_ingreso:04d}",
            "Residuo_Real": residuo,
            "Residuo_Registrado": residuo_registrado,
            "Tipo": tipos_residuos[residuo],
            "Destino": destinos_residuos[residuo],
            "FechaHora_Real": fecha_hora_real,
            "FechaHora_Registrada": fecha_hora_registrada,
            "Cantidad_Real_kg": round(cantidad_real, 2),
            "Cantidad_Registrada_kg": round(cantidad_registrada, 2),
            "Tipo_Error": tipo_error,
            "Descripcion_Error": error_descripcion
        })

        id_ingreso += 1

df_ingresos = pd.DataFrame(ingresos_2025)
df_errores_ingresos = pd.DataFrame(errores_registrados)

# Guardar archivo de ingresos (solo con datos registrados)
df_ingresos[["ID_Evento", "Residuo_Registrado", "FechaHora_Registrada", "Cantidad_Registrada_kg"]].rename(
    columns={"Residuo_Registrado": "Residuo", "FechaHora_Registrada": "FechaHora", "Cantidad_Registrada_kg": "Cantidad_kg"}
).to_csv(f"{INPUT_DIR}/Ingreso_2025.csv", index=False)

df_errores_ingresos.to_csv(f"{INPUT_DIR}/Log_Errores_Ingresos.csv", index=False)

# Calcular errores de subestimación en ingresos
errores_subestimacion = df_errores_ingresos[df_errores_ingresos["Error_kg"] < 0]
print(f"   ✅ Generado: {len(df_ingresos)} ingresos con {len(df_errores_ingresos)} errores registrados")
print(f"   📉 Errores de subestimación en ingresos: {len(errores_subestimacion)} eventos")

# ===============================
# 3. GENERAR SALIDAS 2025 (SIN ERRORES)
# ===============================
print("\n📤 3. Generando Salidas 2025 (SIN ERRORES - datos reales)...")

salidas_2025 = []
id_salida = 1

# Calcular stock disponible por residuo (usando cantidades reales)
stock_disponible = defaultdict(float)
for _, row in df_inv_2024.iterrows():
    stock_disponible[row["Residuo"]] += row["Cantidad_Real_kg"]

for _, ing in df_ingresos.iterrows():
    stock_disponible[ing["Residuo_Real"]] += ing["Cantidad_Real_kg"]

for residuo in residuos:
    stock_total = stock_disponible[residuo]

    # Determinar eficiencia de salida (70-95% del stock)
    eficiencia = random.uniform(0.70, 0.95)
    cantidad_total_salida = stock_total * eficiencia

    # Número de eventos de salida
    num_salidas = random.randint(5, 25)
    proporciones = np.random.dirichlet(np.ones(num_salidas))
    cantidades = proporciones * cantidad_total_salida

    for cantidad in cantidades:
        # SIN ERRORES - los datos registrados son iguales a los reales
        cantidad_registrada = cantidad
        destino_correcto = destinos_residuos[residuo]
        destino_registrado = destino_correcto

        # Decidir de qué inventario se despacha (FIFO: 2024 primero)
        es_inventario_2024 = random.random() < 0.4  # 40% de probabilidad de ser 2024

        salidas_2025.append({
            "ID_Evento": f"SAL-2025-{id_salida:04d}",
            "Residuo": residuo,
            "Tipo": tipos_residuos[residuo],
            "Destino": destino_registrado,
            "FechaHora": generar_fecha_aleatoria_2025(),
            "Cantidad_Real_kg": round(cantidad, 2),
            "Cantidad_Registrada_kg": round(cantidad_registrada, 2),
            "Inventario": "2024" if es_inventario_2024 else "2025"
        })

        id_salida += 1

df_salidas = pd.DataFrame(salidas_2025)

# Guardar archivo de salidas (solo con datos registrados)
df_salidas[["ID_Evento", "Residuo", "FechaHora", "Cantidad_Registrada_kg", "Inventario"]].rename(
    columns={"Cantidad_Registrada_kg": "Cantidad_kg"}
).to_csv(f"{INPUT_DIR}/Salida_2025.csv", index=False)

print(f"   ✅ Generado: {len(df_salidas)} salidas SIN ERRORES")

# ===============================
# 4. CALCULAR INVENTARIO FÍSICO 2025 (REAL, SIN ERRORES)
# ===============================
print("\n📊 4. Calculando Inventario Físico 2025 (real, SIN ERRORES)...")

# Aplicar FIFO físico real para calcular inventario final
inventario_fisico_2025 = []

for residuo in residuos:
    # 1. Recolectar todos los stocks disponibles (ordenados por fecha)
    stocks = []

    # Inventario 2024 (más antiguo)
    inv_2024_res = df_inv_2024[df_inv_2024["Residuo"] == residuo]
    for _, inv in inv_2024_res.iterrows():
        stocks.append({
            "Fecha": inv["FechaHora"],
            "Cantidad": inv["Cantidad_Real_kg"],
            "Tipo": "Inventario_2024",
            "ID": inv["ID_Evento"]
        })

    # Ingresos 2025 (ordenados por fecha real)
    ingresos_res = df_ingresos[df_ingresos["Residuo_Real"] == residuo].sort_values("FechaHora_Real")
    for _, ing in ingresos_res.iterrows():
        stocks.append({
            "Fecha": ing["FechaHora_Real"],
            "Cantidad": ing["Cantidad_Real_kg"],
            "Tipo": "Ingreso_2025",
            "ID": ing["ID_Evento"]
        })

    # Ordenar por fecha (FIFO)
    stocks.sort(key=lambda x: x["Fecha"])

    # 2. Calcular salidas reales
    salidas_res = df_salidas[df_salidas["Residuo"] == residuo].sort_values("FechaHora")
    salidas_totales = salidas_res["Cantidad_Real_kg"].sum()

    # 3. Aplicar FIFO: descontar salidas del stock más antiguo
    saldo_pendiente = salidas_totales

    for stock in stocks:
        if saldo_pendiente <= 0:
            break

        disponible = stock["Cantidad"]
        if disponible > 0:
            usado = min(disponible, saldo_pendiente)
            stock["Cantidad"] -= usado
            saldo_pendiente -= usado

    # 4. Calcular inventario físico final
    inventario_final = sum(stock["Cantidad"] for stock in stocks)

    inventario_fisico_2025.append({
        "Residuo": residuo,
        "Tipo": tipos_residuos[residuo],
        "Destino": destinos_residuos[residuo],
        "FechaHora": datetime(2025, 12, 31, 23, 59),
        "Cantidad_kg": round(inventario_final, 2)
    })

df_inv_fisico = pd.DataFrame(inventario_fisico_2025)
df_inv_fisico.to_csv(f"{INPUT_DIR}/Inventario_Fisico_2025.csv", index=False)
print(f"   ✅ Calculado: Inventario físico para {len(df_inv_fisico)} residuos (SIN ERRORES)")

# ===============================
# 5. CALCULAR INVENTARIO TEÓRICO 2025 (CON MÁS ERRORES DE REGISTRO)
# ===============================
print("\n🧮 5. Calculando Inventario Teórico 2025 (con MÁS errores de registro)...")

inventario_teorico_2025 = []

for residuo in residuos:
    # Inventario 2024 registrado (con errores)
    inv_2024_reg = df_inv_2024[df_inv_2024["Residuo"] == residuo]["Cantidad_Registrada_kg"].sum()

    # Ingresos 2025 registrados (con errores)
    ingresos_reg = df_ingresos[df_ingresos["Residuo_Registrado"] == residuo]["Cantidad_Registrada_kg"].sum()

    # Salidas 2025 registradas (SIN errores)
    salidas_reg = df_salidas[df_salidas["Residuo"] == residuo]["Cantidad_Registrada_kg"].sum()

    # Cálculo teórico: Inventario inicial + Ingresos - Salidas
    inventario_teorico = inv_2024_reg + ingresos_reg - salidas_reg

    inventario_teorico_2025.append({
        "Residuo": residuo,
        "Tipo": tipos_residuos[residuo],
        "Destino": destinos_residuos[residuo],
        "FechaHora": datetime(2025, 12, 31, 23, 59),
        "Cantidad_kg": round(inventario_teorico, 2)
    })

df_inv_teorico = pd.DataFrame(inventario_teorico_2025)
df_inv_teorico.to_csv(f"{INPUT_DIR}/Inventario_Teorico_2025.csv", index=False)
print(f"   ✅ Calculado: Inventario teórico para {len(df_inv_teorico)} residuos (CON ERRORES)")

# ===============================
# 6. CALCULAR DIFERENCIAS Y ANALIZAR PROBLEMAS
# ===============================
print("\n🔍 6. Analizando diferencias y problemas...")

# Comparar inventarios
comparacion = pd.merge(
    df_inv_teorico[["Residuo", "Cantidad_kg"]].rename(columns={"Cantidad_kg": "Teorico"}),
    df_inv_fisico[["Residuo", "Cantidad_kg"]].rename(columns={"Cantidad_kg": "Fisico"}),
    on="Residuo"
)

# Agregar información de tipo y destino
comparacion = comparacion.merge(
    lista_maestra[["Residuo", "Tipo", "Destino"]],
    on="Residuo"
)

# Calcular diferencias
comparacion["Diferencia_kg"] = comparacion["Teorico"] - comparacion["Fisico"]
comparacion["Diferencia_Absoluta_kg"] = comparacion["Diferencia_kg"].abs()
comparacion["%_Diferencia"] = (comparacion["Diferencia_Absoluta_kg"] /
                               comparacion["Fisico"].replace(0, 0.01) * 100)

# Clasificar tipos de problemas
def clasificar_problema(teorico, fisico, diferencia):
    if teorico < 0:
        return "BALANCE NEGATIVO"
    elif diferencia > 100:  # Más de 100 kg de diferencia
        return "TEÓRICO > FÍSICO (SUBESTIMACIÓN)"
    elif diferencia < -100:  # Menos de -100 kg de diferencia
        return "TEÓRICO < FÍSICO (SOBREESTIMACIÓN)"
    elif abs(diferencia) <= 100:
        return "DIFERENCIA MENOR"
    else:
        return "SIN PROBLEMA"

comparacion["Tipo_Problema"] = comparacion.apply(
    lambda x: clasificar_problema(x["Teorico"], x["Fisico"], x["Diferencia_kg"]), axis=1
)

# Calcular estadísticas
estadisticas_problemas = comparacion["Tipo_Problema"].value_counts()

print(f"\n📊 DISTRIBUCIÓN DE PROBLEMAS DE BALANCE (CON MÁS BALANCES NEGATIVOS):")
for problema, cantidad in estadisticas_problemas.items():
    porcentaje = cantidad / len(comparacion) * 100
    print(f"   • {problema}: {cantidad} residuos ({porcentaje:.1f}%)")

# Calcular diferencia total
diferencia_total = comparacion["Diferencia_kg"].sum()
print(f"\n📈 DIFERENCIA TOTAL ACUMULADA: {diferencia_total:+,.0f} kg")

# Identificar balances negativos específicos
balances_negativos = comparacion[comparacion["Teorico"] < 0]
print(f"\n⚠️  BALANCES NEGATIVOS ({len(balances_negativos)} residuos):")
for _, row in balances_negativos.iterrows():
    print(f"   • {row['Residuo']}: {row['Teorico']:,.0f} kg (Físico: {row['Fisico']:,.0f} kg)")

# Identificar residuos críticos
residuos_criticos = comparacion[
    (comparacion["Tipo_Problema"] == "BALANCE NEGATIVO") |
    (comparacion["Diferencia_Absoluta_kg"] > 1000)
].sort_values("Diferencia_Absoluta_kg", ascending=False)

print(f"\n⚠️  RESIDUOS CRÍTICOS ({len(residuos_criticos)}):")
for _, row in residuos_criticos.head(10).iterrows():
    print(f"   • {row['Residuo']}: {row['Diferencia_kg']:+,.0f} kg ({row['Tipo_Problema']})")

# Guardar comparación
comparacion.to_csv(f"{INPUT_DIR}/Comparacion_Inventarios.csv", index=False)

# ===============================
# 7. GENERAR RESUMEN DE ERRORES (SOLO INGRESOS)
# ===============================
print("\n📋 7. Generando resumen de errores (solo ingresos)...")

# Resumen de errores en ingresos
resumen_errores_ingresos = df_errores_ingresos.groupby("Tipo_Error").agg({
    "ID_Ingreso": "count",
    "Error_kg": ["sum", "mean", "max"]
}).round(2)

resumen_errores_ingresos.columns = ["Cantidad", "Total_Error_kg", "Promedio_Error_kg", "Max_Error_kg"]
resumen_errores_ingresos.to_csv(f"{INPUT_DIR}/Resumen_Errores_Ingresos.csv")

print(f"\n📂 ARCHIVOS GENERADOS EN '{INPUT_DIR}/':")
print("   1. Inventario_2024.csv (con MÁS errores de subestimación)")
print("   2. Ingreso_2025.csv (con MÁS errores de subestimación)")
print("   3. Log_Errores_Ingresos.csv")
print("   4. Salida_2025.csv (SIN ERRORES - datos reales)")
print("   5. Inventario_Fisico_2025.csv (SIN ERRORES - datos reales)")
print("   6. Inventario_Teorico_2025.csv (con MÁS errores de registro)")
print("   7. Comparacion_Inventarios.csv")
print("   8. Resumen_Errores_Ingresos.csv")

# ===============================
# 8. RESUMEN EJECUTIVO
# ===============================
print(f"\n{'='*80}")
print("🎯 RESUMEN EJECUTIVO - SITUACIÓN INICIAL (CON MÁS BALANCES NEGATIVOS)")
print(f"{'='*80}")

total_ingresos = df_ingresos["Cantidad_Real_kg"].sum()
total_salidas = df_salidas["Cantidad_Real_kg"].sum()
inventario_inicial = df_inv_2024["Cantidad_Real_kg"].sum()
inventario_final_fisico = df_inv_fisico["Cantidad_kg"].sum()
inventario_final_teorico = df_inv_teorico["Cantidad_kg"].sum()

print(f"\n📈 VOLÚMENES TOTALES:")
print(f"   • Inventario inicial 2024: {inventario_inicial/1000:,.1f} ton")
print(f"   • Ingresos 2025: {total_ingresos/1000:,.1f} ton")
print(f"   • Salidas 2025: {total_salidas/1000:,.1f} ton")
print(f"   • Inventario físico final: {inventario_final_fisico/1000:,.1f} ton")
print(f"   • Inventario teórico final: {inventario_final_teorico/1000:,.1f} ton")
print(f"   • Diferencia total: {(inventario_final_teorico - inventario_final_fisico)/1000:+,.1f} ton")

print(f"\n🔍 PROBLEMAS GENERADOS (CON MÁS BALANCES NEGATIVOS):")
print(f"   • Errores en ingresos: {len(df_errores_ingresos)} eventos")
print(f"   • Problemas en salidas: 0 eventos (SIN ERRORES - datos reales)")
print(f"   • Residuos con balance negativo: {len(balances_negativos)}")
print(f"   • Residuos con diferencias > 100 kg: {(comparacion['Diferencia_Absoluta_kg'] > 100).sum()}")
print(f"   • Error total acumulado: {diferencia_total:+,.0f} kg")

print(f"\n📉 CAUSAS PRINCIPALES DE BALANCES NEGATIVOS:")
print("   1. Subestimación severa en pesaje de ingresos (hasta 50% menos)")
print("   2. Cambios de clasificación no registrados")
print("   3. División de residuos no contabilizada")
print("   4. Errores de registro inicial en inventario 2024")

print(f"\n🎯 OBJETIVO DEL EJERCICIO:")
print("   Aplicar método FIFO Físico para corregir discrepancias, especialmente")
print("   los balances negativos causados por subestimación en ingresos.")

print(f"\n✅ CELDA 1 COMPLETADA EXITOSAMENTE (CON MÁS BALANCES NEGATIVOS)")
print(f"   Proceda a ejecutar la CELDA 2 para análisis y aplicación de FIFO.")